# Lekcija 08 - Vzorec večagentnega oblikovanja


## Nastavitev


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Zakaj večagentni sistemi?

Dejanske naloge, kot je načrtovanje potovanja, vključujejo različna strokovna področja — logistiko, lokalno znanje, proračun in drugo. Posamezni agent, ki poskuša obvladati vse, hitro postane nepraktičen.

Večagentni sistemi to rešujejo s **specializacijo**: vsak agent se osredotoči na eno strokovno področje, s čimer doseže boljše rezultate kot generalist. Prav tako izboljšajo **razširljivost** — lahko dodate nove agente (npr. strokovnjaka za lete, restavratorskega kritika) brez prepisovanja obstoječega delovnega toka. Agenti delujejo skupaj skozi strukturirano cevovod, pri čemer prenašajo kontekst od enega do drugega.


## Ustvarjanje specializiranih agentov


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Izdelava zaporednega poteka dela

`WorkflowBuilder` vam omogoča, da povežete agente v usmerjen graf. Tukaj ustvarimo preprost dvostopenjski potek: **TravelPlanner** pripravi načrt poti, nato ga **TravelConcierge** pregleda in izboljša.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodajanje več agentov v delovni tok

Ena največjih prednosti vzorca večagentnega sistema je, kako enostavno ga je razširiti. Spodaj dodamo agenta **BudgetReviewer**, ki preveri načrt glede na proračun potnika, označi postavke, ki bi lahko povzročile preseganje stroškov, in predlaga denar prihranjujoče alternative. Delovni tok zdaj zaporedoma zažene tri agente:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Povzetek

V tej lekciji ste se naučili:

1. **Ustvariti specializirane agente** — vsak s specifično vlogo (načrtovanje, vratar, pregled proračuna).
2. **Povezati agente v zaporedno delovno tok** z uporabo `WorkflowBuilder` in `add_edge`.
3. **Pretakati izhod** iz cevovoda z več agenti ter spremljati, kateri agent govori.
4. **Razširiti delovni tok** z dodajanjem novih agentov v verigo, ne da bi spreminjali obstoječe.

Vzorčni načrt z več agenti ohranja vsakega agenta preprostega, hkrati pa ustvarja bogatejše in bolj temeljito pregledane rezultate, kot jih lahko doseže posamezen agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Opozorilo**:
Ta dokument je bil preveden z uporabo storitve umetne inteligence za prevajanje [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas opozarjamo, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku velja za avtoritativni vir. Za pomembne informacije priporočamo strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
